In [1]:
import os
import shutil
import zipfile
from pathlib import Path
import pandas as pd
from tqdm import tqdm

In [2]:

root_dir = r'Y:\ZHL\isds\PS\task0730'
merge_dir = os.path.join(root_dir, 'merge_dir')
root_folder_id = '1XzCNgXQo9aOpr_T7PqTgBz0WM8thcL0B'
client_secret = r"E:\data\202502_signboard\data_annotation\docs\client_secret.json"
token_path = r'E:\repository\dataset_tools\isds_tool\PS_data\token.json'
SCOPES = ['https://www.googleapis.com/auth/drive.readonly']

slam_root_folder_id = '1812odFR-dVIN8TwHyCcVlg27bhMsrMoc'
gap_num = 3

In [3]:
# os.remove(token_path)

In [3]:
import os
import io
from concurrent.futures import ThreadPoolExecutor
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request


def authenticate_with_google(token_path, client_secret_path):
    creds = None

    if os.path.exists(token_path):
        creds = Credentials.from_authorized_user_file(token_path, SCOPES)

    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(client_secret_path, SCOPES)
            creds = flow.run_local_server(port=0)
        with open(token_path, 'w') as token_file:
            token_file.write(creds.to_json())

    service = build('drive', 'v3', credentials=creds)
    return service


def download_large_file(service, file_id, file_path):
    os.makedirs(os.path.dirname(file_path), exist_ok=True)
    if os.path.exists(file_path):
        print(f"⚠️ 已存在，跳过: {file_path}")
        return
    print(f"⬇️ Downloading {file_path}")
    request = service.files().get_media(fileId=file_id)
    with io.FileIO(file_path, 'wb') as fh:
        downloader = MediaIoBaseDownload(fh, request)
        done = False
        while not done:
            status, done = downloader.next_chunk()
            if status:
                print(f"⬇️ Downloading {file_path}: {int(status.progress() * 100)}%")
    print(f"✅ Finished: {file_path}")

def download_folder_recursive(service, folder_id, save_path):
    os.makedirs(save_path, exist_ok=True)
    query = f"'{folder_id}' in parents and trashed = false"
    results = service.files().list(q=query, fields="files(id, name, mimeType)").execute()
    items = results.get('files', [])

    for item in items:
        file_id = item['id']
        file_name = item['name']
        file_mime = item['mimeType']
        full_path = os.path.join(save_path, file_name)

        if file_mime == 'application/vnd.google-apps.folder':
            download_folder_recursive(service, file_id, full_path)
        else:
            download_large_file(service, file_id, full_path)

def download_subfolder_task(folder_obj, root_save_path, token_path, client_secret_path):
    # 每个线程都单独认证，避免多线程共享service导致问题
    service = authenticate_with_google(token_path, client_secret_path)
    folder_id = folder_obj['id']
    folder_name = folder_obj['name']
    target_path = os.path.join(root_save_path, folder_name)
    print(f"\n📁 Starting folder: {folder_name}")
    download_folder_recursive(service, folder_id, target_path)

def download_all_subfolders_parallel(token_path, client_secret_path, root_folder_id, save_dir):
    os.makedirs(save_dir, exist_ok=True)

    # 主线程先获取子文件夹列表
    service = authenticate_with_google(token_path, client_secret_path)
    query = f"'{root_folder_id}' in parents and trashed = false and mimeType = 'application/vnd.google-apps.folder'"
    results = service.files().list(q=query, fields="files(id, name)").execute()
    folders = results.get('files', [])

    print(f"将并发下载 {len(folders)} 个子文件夹...\n")

    with ThreadPoolExecutor(max_workers=len(folders)) as executor:
        for folder in folders:
            executor.submit(download_subfolder_task, folder, save_dir, token_path, client_secret_path)



In [4]:
download_all_subfolders_parallel(token_path, client_secret, root_folder_id, root_dir)

将并发下载 4 个子文件夹...


📁 Starting folder: syp1

📁 Starting folder: ymt1

📁 Starting folder: ymt2

📁 Starting folder: syp2
⬇️ Downloading Y:\ZHL\isds\PS\task0730\ymt2\gps_data_20250730132929900_20250730134038500.txt
⬇️ Downloading Y:\ZHL\isds\PS\task0730\ymt1\gps_data_20250730145022500_20250730145825399.txt
⬇️ Downloading Y:\ZHL\isds\PS\task0730\syp2\imu_data_20250730153526327_20250730154733545.txt
⬇️ Downloading Y:\ZHL\isds\PS\task0730\syp1\gps_data_20250730145022500_20250730145825399.txt
⬇️ Downloading Y:\ZHL\isds\PS\task0730\ymt1\gps_data_20250730145022500_20250730145825399.txt: 100%
⬇️ Downloading Y:\ZHL\isds\PS\task0730\syp1\gps_data_20250730145022500_20250730145825399.txt: 100%
⬇️ Downloading Y:\ZHL\isds\PS\task0730\ymt2\gps_data_20250730132929900_20250730134038500.txt: 100%
✅ Finished: Y:\ZHL\isds\PS\task0730\ymt1\gps_data_20250730145022500_20250730145825399.txt✅ Finished: Y:\ZHL\isds\PS\task0730\syp1\gps_data_20250730145022500_20250730145825399.txt

✅ Finished: Y:\ZHL\isds\PS\task07

In [9]:
def uzip_dirs(root_dir):
    sub_dir_list = os.listdir(root_dir)
    for sub_dir_name in sub_dir_list:
        sub_dir = os.path.join(root_dir, sub_dir_name)
        zip_path = os.path.join(sub_dir, 'rectified_images.zip')
        if not os.path.exists(zip_path):
            print(f'{zip_path} not exists')
        else:
            print(f'{zip_path} unzip...')
            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                zip_ref.extractall(zip_path.replace('.zip', ''))
            print(f'{zip_path} done\n')

In [8]:
uzip_dirs(root_dir)

Y:\ZHL\isds\PS\task0801\12-08-40\rectified_images.zip not exists
Y:\ZHL\isds\PS\task0801\13-18-22\rectified_images.zip not exists
Y:\ZHL\isds\PS\task0801\13-53-35\rectified_images.zip not exists
Y:\ZHL\isds\PS\task0801\15-33-39\rectified_images.zip not exists
Y:\ZHL\isds\PS\task0801\16-17-20\rectified_images.zip not exists
Y:\ZHL\isds\PS\task0801\16-34-56\rectified_images.zip not exists
Y:\ZHL\isds\PS\task0801\results\rectified_images.zip not exists


In [ ]:
from img_preprocess import select_img
from deduplication_demo import filter_deduplication

def process_dirs(root_dir):
    sub_dirs = os.listdir(root_dir)
    for idx, sub_name in enumerate(sub_dirs):
        sub_dir = os.path.join(root_dir, sub_name)
        if not os.path.isdir(sub_dir) or not sub_name.startswith('y'):
            continue
        cam_name_list = ['cam_DA4930148', 'cam_DA5148680', 'cam_DA5148683', 'cam_DA5324645', 'cam_DA5324655', 'cam_DA6102933']
        for cam_name in cam_name_list:
            image_dir_src = os.path.join(sub_dir, 'rectified_images', 'rectified_images', cam_name)
            if not os.path.exists(image_dir_src):
                print(f'{image_dir_src} not exists')
            else:
                print(f'{image_dir_src} selecting...')
                image_dir_select = image_dir_src+'_select'
                shutil.rmtree(image_dir_select) if os.path.exists(image_dir_select) else None
                select_img(image_dir_src, image_dir_select, gap=gap_num)
                print(f'{image_dir_select} filtering...')
                image_dir_filter = image_dir_src+'_filter' 
                shutil.rmtree(image_dir_filter) if os.path.exists(image_dir_filter) else None
                filter_deduplication(image_dir_select, image_dir_filter)
                print(f'{image_dir_filter} done\n')

In [19]:
process_dirs(root_dir)

Y:\ZHL\isds\PS\task0725\ymt--1\rectified_images\rectified_images\cam_DA4930148 selecting...


100%|██████████| 131/131 [00:04<00:00, 27.57it/s]


Y:\ZHL\isds\PS\task0725\ymt--1\rectified_images\rectified_images\cam_DA4930148_select filtering...


100%|██████████| 99/99 [00:02<00:00, 40.76it/s]



Total unique images copied: 99
Y:\ZHL\isds\PS\task0725\ymt--1\rectified_images\rectified_images\cam_DA4930148_filter done

Y:\ZHL\isds\PS\task0725\ymt--1\rectified_images\rectified_images\cam_DA5148680 selecting...


...


Total unique images copied: 156
Y:\ZHL\isds\PS\task0725\ymt-2\rectified_images\rectified_images\cam_DA5324655_filter done

Y:\ZHL\isds\PS\task0725\ymt-2\rectified_images\rectified_images\cam_DA6102933 selecting...


100%|██████████| 369/369 [00:07<00:00, 51.26it/s]


Y:\ZHL\isds\PS\task0725\ymt-2\rectified_images\rectified_images\cam_DA6102933_select filtering...


100%|██████████| 176/176 [00:04<00:00, 39.34it/s]


Total unique images copied: 176
Y:\ZHL\isds\PS\task0725\ymt-2\rectified_images\rectified_images\cam_DA6102933_filter done



In [20]:
def img_merge(input_dir, output_dir):
    sub_dirs = os.listdir(input_dir)
    if 'merge_dir' in sub_dirs:
        sub_dirs.remove('merge_dir')
    os.makedirs(output_dir, exist_ok=True)
    for sub_name in sub_dirs:
        sub_dir = os.path.join(input_dir, sub_name)
        if not os.path.isdir(sub_dir):
            continue
        cam_name_list = ['cam_DA4930148', 'cam_DA5148680', 'cam_DA5148683', 'cam_DA5324645', 'cam_DA5324655', 'cam_DA6102933']
        for cam_name in cam_name_list:
            image_dir_src = os.path.join(sub_dir, 'rectified_images', 'rectified_images', cam_name+'_filter')
            if not os.path.exists(image_dir_src):
                print(f'{image_dir_src} not exists')
            else:
                img_list = os.listdir(image_dir_src)
                for img_name in tqdm(img_list):
                    img_path_src = os.path.join(image_dir_src, img_name)
                    img_path_dst = os.path.join(output_dir, cam_name+'_'+img_name)
                    shutil.copyfile(img_path_src, img_path_dst)
    


In [21]:
img_merge(root_dir, merge_dir)

Y:\ZHL\isds\PS\task0725\results\rectified_images\rectified_images\cam_DA4930148_filter not exists
Y:\ZHL\isds\PS\task0725\results\rectified_images\rectified_images\cam_DA5148680_filter not exists
Y:\ZHL\isds\PS\task0725\results\rectified_images\rectified_images\cam_DA5148683_filter not exists
Y:\ZHL\isds\PS\task0725\results\rectified_images\rectified_images\cam_DA5324645_filter not exists
Y:\ZHL\isds\PS\task0725\results\rectified_images\rectified_images\cam_DA5324655_filter not exists
Y:\ZHL\isds\PS\task0725\results\rectified_images\rectified_images\cam_DA6102933_filter not exists


  0%|          | 0/99 [00:00<?, ?it/s]

100%|██████████| 176/176 [00:03<00:00, 52.64it/s]


In [22]:
print(len(os.listdir(merge_dir)))

1355


In [23]:
import zipfile
import os

def zip_folder_to_path(source_folder, destination_zip):
    with zipfile.ZipFile(destination_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(source_folder):
            for file in files:
                file_path = os.path.join(root, file)
                # 在zip文件中创建相对路径
                arcname = os.path.relpath(file_path, start=source_folder)
                zipf.write(file_path, arcname)
    
    print(f"zip '{source_folder}' to '{destination_zip}'")

zip_folder_to_path(
    source_folder=merge_dir,
    destination_zip=os.path.join(root_dir, os.path.basename(root_dir)+'.zip')
)

zip 'Y:\ZHL\isds\PS\task0725\merge_dir' to 'Y:\ZHL\isds\PS\task0725\task0725.zip'


In [ ]:
download_all_subfolders_parallel(token_path, client_secret, slam_root_folder_id, root_dir)

将并发下载 5 个子文件夹...


📁 Starting folder: 12-22-22

📁 Starting folder: 13-31-34

📁 Starting folder: 14-31-37

📁 Starting folder: 15-14-55

📁 Starting folder: 11-01-54
⬇️ Downloading Y:\ZHL\isds\PS\task0815\13-31-34\slam_geo_referenced.txt
⬇️ Downloading Y:\ZHL\isds\PS\task0815\12-22-22\slam_geo_referenced.txt
⬇️ Downloading Y:\ZHL\isds\PS\task0815\14-31-37\slam_geo_referenced.txt
⬇️ Downloading Y:\ZHL\isds\PS\task0815\15-14-55\slam_geo_referenced.txt
⬇️ Downloading Y:\ZHL\isds\PS\task0815\11-01-54\slam_geo_referenced.txt
⬇️ Downloading Y:\ZHL\isds\PS\task0815\15-14-55\slam_geo_referenced.txt: 100%
✅ Finished: Y:\ZHL\isds\PS\task0815\15-14-55\slam_geo_referenced.txt
⬇️ Downloading Y:\ZHL\isds\PS\task0815\15-14-55\geo_ref_matrix.txt
⬇️ Downloading Y:\ZHL\isds\PS\task0815\14-31-37\slam_geo_referenced.txt: 100%
✅ Finished: Y:\ZHL\isds\PS\task0815\14-31-37\slam_geo_referenced.txt
⬇️ Downloading Y:\ZHL\isds\PS\task0815\14-31-37\geo_ref_matrix.txt
⬇️ Downloading Y:\ZHL\isds\PS\task0815\11-01-54\s